# 4. Plots

RadDB draws four kinds of radar plot:

| method | name | what it shows |
|---|---|---|
| `plot_ppi(sweep=...)` | Plan Position Indicator | one sweep seen from above — a map |
| `plot_rhi(azimuth=...)` | Range Height Indicator | one azimuth seen from the side — a vertical slice along one ray |
| `plot_cappi(altitude=...)` | Constant Altitude PPI | one altitude seen from above — a horizontal slice through the volume |
| `plot_vcs(line=...)` | Vertical Cross-Section | a vertical slice along any line you choose |

Each one draws into **one Axes** and returns the matplotlib artist, so you build
multi-panel figures yourself by passing `ax=` (section 7).

They plot exactly the gates the data holds: filter it, crop it or `sel` it first
and the plot follows, gate for gate.

---

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt

import raddb

# Keep the embedded figures small enough for GitHub to render this notebook.
# plt.rcParams["figure.dpi"] = 70

In [ ]:
# --------------------------------------------------------------------------
# CONFIGURATION — edit these three paths to point at your own data
# --------------------------------------------------------------------------
# ARCHIVE_DIR must be the same archive tutorial 1 wrote. If it has not run,
# the cell below builds it.

MCH_DIR = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/MCH_datatree").expanduser()
NEXRAD_DIR = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/NEXRAD_datatree").expanduser()
ARCHIVE_DIR = Path("~/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive").expanduser()

print("MCH DataTrees   :", MCH_DIR)
print("NEXRAD DataTrees:", NEXRAD_DIR)
print("Archive         :", ARCHIVE_DIR)

In [ ]:
# This notebook stands on its own: build the archive if tutorial 1 has not run.
if not (ARCHIVE_DIR / "L" / "LUT").exists():
    print("building the archive (see tutorial 1) ...")
    raddb.RadDB(archive_dir=ARCHIVE_DIR, crs=2056).archive(datatree_dir=MCH_DIR)
else:
    print("archive already present:", ARCHIVE_DIR)

In [ ]:
db = raddb.RadDB(archive_dir=ARCHIVE_DIR)
rdf = db.open(radars="L")
info = db.get_radar_info("L")
SITE = (info["longitude"], info["latitude"])
print(f"{len(rdf):,} gates | variables: {rdf.columns()}")

In [ ]:
# ==================USING NEXRAD DATA=========================
db = raddb.RadDB(archive_dir=ARCHIVE_DIR)
rdf = db.open(radars="KTLX")
info = db.get_radar_info("KTLX")
SITE = (info["longitude"], info["latitude"])
print(f"{len(rdf):,} gates | variables: {rdf.columns()}")

## 0. Which timesteps can I plot?

Every plot draws **one volume**. If `rdf` holds more than one, you must say which
with `timestep=`, otherwise the call raises rather than drawing them on top of
each other:

```
ValueError: data holds 7 volumes (2024-06-08 09:45:03 ... 2024-06-12 00:40:05);
pass timestep= to pick one, or narrow with start_time=/end_time=.
```

The list is in the data itself — `volume_time` is a normal column:

In [ ]:
TIMESTEPS = rdf.data["volume_time"].unique().sort().to_list()
print(f"{len(TIMESTEPS)} volumes in rdf:")
for t in TIMESTEPS:
    print("  ", t)

In [ ]:
df = rdf.filter({"var": "DBZH", "logic": ">", "threshold": 20})
print(f"{len(rdf):,} gates -> {len(df):,} above 30 dBZ")
df.to_pandas().describe()

`timestep=` takes anything pandas reads as a time and picks the **nearest**
volume, so you can be as loose or as exact as you like:

```python
rdf.plot_ppi(sweep=1, timestep="2024-06-12")            # nearest to midnight -> 00:30:08
rdf.plot_ppi(sweep=1, timestep="2024-06-12 00:35")      # nearest to 00:35    -> 00:35:01
rdf.plot_ppi(sweep=1, timestep=TIMESTEPS[-1])           # exactly that volume
```

Two other ways to get to a single volume:

- `start_time=` / `end_time=` narrow the candidates first — if only one is left,
  `timestep=` is not needed at all;
- `rdf.sel(volume_time=TIMESTEPS[0])` (tutorial 2) returns a RadDB holding just
  that volume, which then plots with no time argument at all.

The rest of this notebook uses `timestep="2024-06-12"`.

## 1. PPI

In [ ]:
fig, ax = plt.subplots()
rdf.plot_ppi(sweep=11, variable="DBZH", timestep="2024-06-12", ax=ax, coords="projected", context=True)
plt.tight_layout()
plt.show()

## 2. RHI

In [ ]:
fig, ax = plt.subplots()
rdf.plot_rhi(azimuth=90, variable="DBZH", timestep="2024-06-12", ax=ax)
plt.tight_layout()
plt.show()

## 3. CAPPI

In [ ]:
fig, ax = plt.subplots()
rdf.plot_cappi(altitude=2000, variable="DBZH", timestep="2024-06-12", ax=ax)
plt.tight_layout()
plt.show()

## 4. Vertical cross-section

A PPI has its sweep and an RHI its azimuth; a cross-section needs a **line**.
Either cut it first with `extract_cross_section` (tutorial 3) and then draw it:

In [ ]:
cs = rdf.extract_cross_section(p1=(SITE[0] - 0.6, SITE[1] - 0.35), p2=(SITE[0] + 0.6, SITE[1] + 0.35), crs=4326)

fig, ax = plt.subplots(figsize=(8, 4.5))
cs.plot_vcs(variable="DBZH", timestep="2024-06-12", ax=ax)
plt.tight_layout()
plt.show()

... or hand the line straight to `plot_vcs`, which cuts and draws in one step:

```python
rdf.plot_vcs(line=[(lon1, lat1), (lon2, lat2)], crs=4326)
rdf.plot_vcs(line="my_section.geojson")           # a LineString from a file
```

Passing a line to something that **already** carries a section is an error, and so
is drawing a section from data that has none — a cross-section has to be defined
exactly once.

### 4.1 Drawing the section on the map

Typing lon/lat pairs is fine when you know where the storm is. When you don't,
draw the line instead: `interactive_crop()` (tutorial 3) puts an ipyleaflet map in
front of you, and a **polyline** is dispatched to `extract_cross_section`.

So the whole chain is: draw → *Apply crop* → `selector.result` → `plot_vcs()`.
The result already carries `cs_polygon`, so `plot_vcs` needs **no** `line=`.

In [ ]:
RUN_INTERACTIVE = True  # <- set False to skip the map (e.g. outside Jupyter)

if RUN_INTERACTIVE:
    # Pick the "/" polyline tool in the toolbar, draw a line across the echo,
    # then click "Apply crop".  Only then run the next cell.
    selector = rdf.sel(volume_time=TIMESTEPS[4]).interactive_crop()
else:
    selector = None
    print("needs a live Jupyter kernel ==> set RUN_INTERACTIVE = True")

In [ ]:
drawn = getattr(selector, "result", None)

if drawn is None:
    print("nothing applied yet — draw a line above and click 'Apply crop'.")
elif selector.kind != "cross_section":
    print(f"you drew a {selector.kind!r}, not a line; " "only the polyline tool produces a cross-section.")
else:
    print(f"{len(drawn):,} gates on the drawn section")
    fig, ax = plt.subplots(figsize=(8, 4.5))
    drawn.plot_vcs(variable="DBZH", ax=ax)
    plt.tight_layout()
    plt.show()

## 5. Coordinates and Context

`coords` controls the horizontal frame of the map-like plots:

| value | axes |
|---|---|
| `"xy"` *(default)* | metres east/north of the radar |
| `"lonlat"` | degrees |
| `"projected"` | the archive's own CRS (`x_2056` / `y_2056` here) |
| an EPSG int | that projection |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, coords in zip(axes, ["xy", "lonlat", "projected"], strict=False):
    rdf.plot_ppi(sweep=1, ax=ax, coords=coords, timestep="2024-06-12")
    ax.set_title(f"coords={coords!r}")
plt.tight_layout()
plt.show()

`context=True` overlays country borders and coastlines. Projection and backdrop
are independent: you choose the frame with `coords`, the backdrop with `context`,
and the borders are reprojected into whichever frame you picked.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, coords in zip(axes, ["xy", "lonlat", "projected"], strict=False):
    rdf.plot_ppi(sweep=1, ax=ax, coords=coords, context=True, timestep="2024-06-12")
    ax.set_title(f"coords={coords!r}")
plt.tight_layout()
plt.show()

### 5.1 Choosing the extent

`xlim=` / `ylim=` are ordinary arguments of all four plots — not `**plot_kwargs`,
which go to the colouring. Each takes a `(min, max)` pair **in the units of the
frame you asked for**:

| `coords` | units of `xlim` / `ylim` |
|---|---|
| `"xy"` | metres from the radar (negative to the west/south) |
| `"lonlat"` | degrees |
| `"projected"` / an EPSG int | metres in that projection (LV95 is ~2.6e6 / 1.2e6) |

The tick *labels* are in km, the numbers you pass are in metres — that is why
`xlim=(-50_000, 50_000)` shows an axis running −50 to 50.

Left alone, a PPI/CAPPI frames a square centred on the radar, sized by the
furthest gate drawn. `plot_rhi` also accepts `max_range_km` / `max_height_km` as a
friendlier spelling of the same thing.

`ax.set_xlim(...)` after the call works too — the plot is plain matplotlib.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# same sweep, zoomed to 60 km around the radar
rdf.plot_ppi(
    sweep=3,
    ax=axes[0],
    coords="xy",
    context=True,
    timestep="2024-06-12",
    xlim=(-60_000, 60_000),
    ylim=(-60_000, 60_000),
)
axes[0].set_title("coords='xy' | ±60 km")

# the same window written in degrees
rdf.plot_ppi(
    sweep=3,
    ax=axes[1],
    coords="lonlat",
    context=True,
    timestep="2024-06-12",
    xlim=(SITE[0] - 0.8, SITE[0] + 0.8),
    ylim=(SITE[1] - 0.55, SITE[1] + 0.55),
)
axes[1].set_title("coords='lonlat' | same box but in degrees")

plt.tight_layout()
plt.show()

## 6. Any variable, any subset

`variable=` takes any column the data holds — including one you computed with
`add_feature` (tutorial 2).

In [ ]:
VARS = [v for v in ("DBZH", "ZDR", "RHOHV", "PHIDP") if v in rdf.columns()]

# Pick a sweep that actually carries the dual-pol moments.  On NEXRAD the low
# tilts are "split cuts": the odd sweeps are the Doppler half and hold DBZH only,
# so plotting ZDR there raises "every 'ZDR' value is NaN".
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, var in zip(axes.flat, VARS, strict=False):
    rdf.plot_ppi(sweep=2, variable=var, ax=ax, timestep="2024-06-12", context=True, coords="xy")
    ax.set_title(var)
plt.tight_layout()
plt.show()

In [ ]:
# Filter and crop first — the plot follows the data, gate for gate
sub = rdf.filter({"var": "DBZH", "logic": ">", "threshold": 30}).crop_around_point(
    point=SITE,
    distance=50_000,
    crs=4326,
)

fig, ax = plt.subplots()
art = sub.plot_ppi(sweep=1, ax=ax, timestep="2024-06-12", coords="xy", context=True)
ax.set_title("DBZH > 30 dBz within 50 km")
plt.tight_layout()
plt.show()

## 7. Composing a figure

Because each method fills one Axes, a multi-panel figure is ordinary matplotlib.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.5, 9))
rdf.plot_ppi(sweep=1, ax=axes[0][0], timestep="2024-06-12", title="PPI | sweep 1")
rdf.plot_rhi(azimuth=90, ax=axes[0][1], timestep="2024-06-12", title="RHI | azimuth 90°")
rdf.plot_cappi(altitude=3000, ax=axes[1][0], timestep="2024-06-12", title="CAPPI | altitude 3 km")
cs.plot_vcs(ax=axes[1][1], timestep="2024-06-12", title="cross-section")
fig.suptitle(f"radar {rdf.radars()[0]} | DBZH | 2024-06-12")
plt.tight_layout()
plt.show()

## 8. Plotting without an archive

The same functions accept a raw **DataTree**, computing the geometry from its own
coordinates. Handy for a quick look at a volume you have not archived yet.

The exception is `plot_vcs`: the cross-section path is `gate_id`-keyed, so it
needs an archive.

In [ ]:
dt = raddb.open_any_datatree(sorted(MCH_DIR.glob("L_*.zarr"))[0])

fig, ax = plt.subplots(figsize=(6.2, 5.4))
raddb.plot_ppi(dt, sweep=4, variable="DBZH", ax=ax)
ax.set_title("straight from a DataTree — no archive")
plt.tight_layout()
plt.show()

## 9. Saving

Pass `save="path.png"`, or use matplotlib directly. For vector output on a big
sweep, `rasterized=True` keeps the polygons as pixels inside the PDF and the file
small.

In [ ]:
out = ARCHIVE_DIR / "ppi_example.png"
fig, ax = plt.subplots(figsize=(6.2, 5.4))
rdf.plot_ppi(sweep=1, ax=ax, rasterized=True, timestep="2024-06-12")
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.close(fig)
print("saved:", out, f"({out.stat().st_size / 1e3:.0f} kB)")